# NDVI: estación seca vs. lluviosa

**Material del Webinar introductorio a Google Earth Engine (GEE)**
**Autoría:** Grettel Vargas

**Sentinel-2 → mediana lluviosa → NDVI lluvioso**
**Diferencia = NDVI lluvioso − NDVI seco**

> Es un ejercicio . La mediana ayuda a obtener una imagen representativa de cada período y reduce la influencia de valores extremos.


## 1. Librerías
Carga las librerías necesarias para trabajar con **Google Earth Engine**, visualizar mapas y exportar resultados.


In [ ]:
# Si geemap no está instalado, quitar el # de la siguiente línea y ejecutar esta celda.
# %pip install geemap


In [ ]:
import ee
import geemap

print("✓ Librerías cargadas")


## 2. Parámetros
Define los valores principales del análisis:

- `PAIS` y `DEPTO`: área de estudio.
- `ANIO`: año que se analizará.
- `SECA`: meses utilizados para representar la época seca.
- `LLUVIOSA`: meses utilizados para representar la época lluviosa.
- `MAX_NUBES`: porcentaje máximo de nubosidad permitido.
- `ESCALA_M`: resolución de salida en metros.
- `CRS_SALIDA`: sistema de coordenadas del GeoTIFF.


In [ ]:
PROYECTO_GEE = 'ee-gretapuravida'

PAIS = 'Guatemala'
DEPTO = 'Zacapa'

ANIO = 2025
SECA = (2, 4) # febrero-abril
LLUVIOSA = (9, 10) # septiembre-octubre
MAX_NUBES = 40

ESCALA_M = 30
CRS_SALIDA = 'EPSG:32616'


## 3. Conectar con Earth Engine
Inicia la conexión con **Google Earth Engine** utilizando el proyecto definido en `PROYECTO_GEE`.


In [ ]:
try:
 ee.Initialize(project=PROYECTO_GEE)
except Exception:
 ee.Authenticate()
 ee.Initialize(project=PROYECTO_GEE)

print("✓ Google Earth Engine conectado")


## 4. Área de estudio
Selecciona el departamento indicado en los parámetros utilizando los límites administrativos de **FAO GAUL**.


In [ ]:
limites = ee.FeatureCollection('FAO/GAUL/2015/level1')

depto = limites.filter(
 ee.Filter.And(
 ee.Filter.eq('ADM0_NAME', PAIS),
 ee.Filter.eq('ADM1_NAME', DEPTO)
 )
)

geom = depto.geometry()
lon, lat = geom.centroid().coordinates().getInfo()

print(f"✓ Área de estudio: {DEPTO}, {PAIS}")


## 5. Crear una mediana para cada época
Filtra las imágenes Sentinel-2 para los meses definidos como **seca** y **lluviosa**.

`median()` genera una imagen representativa de cada época calculando la mediana para cada píxel.


In [ ]:
def mediana_epoca(meses):
 coleccion = (
 ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
 .filterBounds(geom)
 .filterDate(f'{ANIO}-01-01', f'{ANIO + 1}-01-01')
 .filter(ee.Filter.calendarRange(meses[0], meses[1], 'month'))
 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_NUBES))
 )

 print(f"Meses {meses[0]}-{meses[1]}: {coleccion.size().getInfo()} imágenes")
 return coleccion.median().clip(geom)


s2_seca = mediana_epoca(SECA)
s2_lluviosa = mediana_epoca(LLUVIOSA)

print("✓ Compuestos por mediana creados")


## 6. Calcular NDVI y diferencia
Calcula el NDVI para cada época usando `B8` y `B4`.

Luego calcula `dNDVI` como:

**NDVI lluviosa − NDVI seca**


In [ ]:
ndvi_seca = s2_seca.normalizedDifference(['B8', 'B4']).rename('NDVI_seca')
ndvi_lluviosa = s2_lluviosa.normalizedDifference(['B8', 'B4']).rename('NDVI_lluviosa')

dndvi = ndvi_lluviosa.subtract(ndvi_seca).rename('dNDVI')

print("✓ NDVI seco, NDVI lluvioso y diferencia calculados")


## 7. Visualizar
Muestra el **NDVI de la época seca**, el **NDVI de la época lluviosa** y su **diferencia** sobre Esri World Imagery.

**Paleta del NDVI (`red → yellow → green`):**
- **Rojo:** valores bajos de NDVI, asociados con poca o ninguna vegetación.
- **Amarillo:** valores intermedios de NDVI.
- **Verde:** valores altos de NDVI, asociados con mayor cobertura o vigor de la vegetación.

**Paleta de la diferencia (`red → white → green`):**
- **Rojo:** el NDVI disminuyó en la época lluviosa respecto a la seca.
- **Blanco:** poco o ningún cambio entre ambas épocas.
- **Verde:** el NDVI aumentó en la época lluviosa respecto a la seca.

Los parámetros `min`, `max` y `palette` definen el rango y los colores utilizados para visualizar los valores.


In [ ]:
vis_ndvi = {
 'min': 0,
 'max': 0.8,
 'palette': ['red', 'yellow', 'green']
}

Map = geemap.Map(
 center=[lat, lon],
 zoom=9,
 basemap='Esri.WorldImagery'
)

Map.addLayer(ndvi_seca, vis_ndvi, 'NDVI - seca')
Map.addLayer(ndvi_lluviosa, vis_ndvi, 'NDVI - lluviosa')
Map.addLayer(
 dndvi,
 {'min': -0.3, 'max': 0.5, 'palette': ['red', 'white', 'green']},
 'Diferencia NDVI'
)
Map.addLayer(depto.style(color='black', fillColor='00000000'), {}, 'Límite')

Map


## 8. Exportar los tres resultados
Exporta tres archivos **GeoTIFF**:

- NDVI de la época seca.
- NDVI de la época lluviosa.
- Diferencia entre ambas épocas (`dNDVI`).


In [ ]:
productos = {
 'NDVI_seca': ndvi_seca,
 'NDVI_lluviosa': ndvi_lluviosa,
 'dNDVI': dndvi
}

for nombre, imagen in productos.items():
 ruta = f"/content/{DEPTO}_{nombre}.tif"

 geemap.ee_export_image(
 imagen.toFloat(),
 filename=ruta,
 scale=ESCALA_M,
 crs=CRS_SALIDA,
 region=geom,
 file_per_band=False
 )

 print("✓", ruta)
